## Ephemeral volume types — `emptyDir`, `hostPath`, `projected`

**`emptyDir` — scratch shared in the pod.** A fresh empty directory created when the pod is scheduled, deleted when the pod is removed. *Container* restarts keep it; *pod* rescheduling wipes it.

```yaml
volumes:
  - name: scratch
    emptyDir: {}                 # node disk
  - name: ramdisk
    emptyDir: { medium: Memory, sizeLimit: 512Mi }   # tmpfs, counts against memory limits
```

For sidecar IPC (the log-shipper), fast per-pod scratch, buffers that don't need durability.

**`hostPath` — a node path mounted into the pod.** A footgun: it punches through isolation, exposes node internals, and makes the pod non-portable. Legitimate uses are narrow — **node-level DaemonSet agents** that must read host paths (log shippers reading `/var/log`, metrics reading `/proc`, CNI writing `/etc/cni/net.d`) and single-node dev.

```yaml
volumes:
  - name: host-logs
    hostPath: { path: /var/log, type: Directory }
```

For anything else, including persistence, use a PV — `hostPath` ties data to one node, and the next schedule loses it.

**`projected` — several sources into one mount.** Combine ConfigMaps, Secrets, and downward-API items into the *same directory* (an app that wants all its config under `/etc/app/`):

```yaml
volumes:
  - name: combined
    projected:
      sources:
        - configMap: { name: app-config }
        - secret: { name: db-credentials }
```

Plus the **ConfigMap / Secret / downwardAPI volumes** from notebook 05 — they're ephemeral too, materialised by the kubelet at mount time. On our map these are all Pod-local volumes — the ephemeral cousins of the durable **Storage** box we turn to next.